# Notebook 1. Contrastive Learning: SimCLR on UCR Time Series

![Status](https://img.shields.io/static/v1.svg?label=Status&message=Finished&color=green)

**Filled notebook:**
[![Open In Collab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lstival/ssl_tutorial_sibgrapi2026/blob/main/notebooks/time_series/01_contrastive_simclr.ipynb)
**Pretrained checkpoints:**
[![GitHub Releases](https://img.shields.io/static/v1.svg?logo=github&label=Repo&message=Checkpoints&color=lightgrey)](https://github.com/lstival/ssl_tutorial_sibgrapi2026/releases)

**Tutorial:** SIBGRAPI 2026 -- *Self-Supervised Learning: Contrastive, Masking, and Distillation Methods*
**Segment:** 3.2 -- Contrastive learning (time series) - Instantiates the lecture part, Section 1.4

---

This is the time-series counterpart of the remote-sensing `01_contrastive_simclr.ipynb`. **Same
mechanism, same loss, same core cell, and now the same downstream task** -- only the data and
the augmentations change.

**Mechanism: instance discrimination.** Two augmented views of the same series are pulled
together in embedding space; every other series in the batch is pushed apart. No labels, no
teacher -- the only signal is "these two views come from the same series."

For time series this is essentially the idea behind **TS2Vec** (Yue et al., 2022) and
**TF-C** (Zhang et al., 2022): contrastive pretraining with time-series-appropriate
augmentations.

### The InfoNCE loss

For a batch of $N$ series we draw two augmented views of each, giving $2N$ views. With $z_i$
the projected embedding of view $i$ and a positive pair $(i,j)$:

$$
\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k)/\tau)}
$$

where $\text{sim}(z_i, z_j) = z_i^\top z_j / (\|z_i\|\,\|z_j\|)$ and $\tau$ is the temperature.
Identical to the remote sensing part -- the mechanism does not care whether $z$ came from an image or a
series.

In [ ]:
## Standard libraries
import os
# Guard against duplicate OpenMP runtimes in some conda envs (see tutorial_ts.py).
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

## Imports for plotting
import matplotlib.pyplot as plt
import numpy as np

## tqdm for loading bars
from tqdm.auto import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim

%matplotlib inline

In [ ]:
# Make the shared tutorial module importable.
# Local clone: it lives in src/time_series/. Colab: download it next to this notebook.
import os, sys, urllib.request

_LOCAL_SRC = os.path.join("..", "..", "src", "time_series")
if os.path.isfile(os.path.join(_LOCAL_SRC, "tutorial_ts.py")):
    sys.path.insert(0, _LOCAL_SRC)
elif not os.path.isfile("tutorial_ts.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lstival/ssl_tutorial_sibgrapi2026/main/src/time_series/tutorial_ts.py", "tutorial_ts.py")

from tutorial_ts import (
    seed_everything, get_device, setup_plotting,
    TARGET_DATASET, SERIES_LEN,
    UCRDataset, UCRCorpusDataset, ucr_class_names,
    ContrastiveTransformations, build_ts_augmentations,
    build_ts_encoder, try_load_checkpoint, plot_curve,
    extract_features, linear_probe_accuracy, plot_embedding_scatter,
)

setup_plotting()
seed_everything(42)
device = get_device()
print("Device:", device)

DATA_PATH = "../../data"
CHECKPOINT_PATH = "../../artifacts/time_series/checkpoints"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

In [ ]:
# --- Check that the pretrained encoders are reachable -------------------------------
# Weights are published on a GitHub Release. Locally they are already on disk once fetched; on Colab they
# are downloaded on demand. This cell just reports what is available, so a missing or
# mis-served checkpoint shows up here rather than halfway through training below.
import os, urllib.request, urllib.error

_CKPTS = ["contrastive_ts_encoder.pt", "mae_ts_encoder.pt", "dino_ts_encoder.pt"]
_REPO = "lstival/ssl_tutorial_sibgrapi2026"
_MEDIA = f"https://github.com/{_REPO}/releases/download/weights-v1/"

for _name in _CKPTS:
    _local = os.path.join(CHECKPOINT_PATH, _name)
    _repo_copy = os.path.join("..", "..", "artifacts", "time_series", "checkpoints", _name)
    _found = next((p for p in (_local, _repo_copy) if os.path.isfile(p)), None)
    if _found and os.path.getsize(_found) > 1024:
        print(f"  on disk    {_name:<26} {os.path.getsize(_found)/1048576:.1f} MB")
        continue
    try:
        _req = urllib.request.Request(_MEDIA + _name, method="HEAD")
        with urllib.request.urlopen(_req, timeout=20) as _r:
            _mb = int(_r.headers.get("Content-Length") or 0) / 1048576
        print(f"  downloadable {_name:<24} {_mb:.1f} MB")
    except Exception as _e:
        print(f"  UNAVAILABLE  {_name:<24} {_e} -> this notebook will train live instead")


## Data: two independent views per series

We reuse the time-series augmentation pipeline from Notebook 0 (jitter, scale/shift,
time-mask, crop-resize -- no flips or reversal) and wrap it in `ContrastiveTransformations`
so each series yields **two** independently augmented views.

The live-training demo below runs on the pooled UCR corpus, and so did the hosted checkpoint --
the only difference is how long it ran (a few hundred steps here versus 15,000 offline).

In [ ]:
ts_aug = build_ts_augmentations()
train_corpus = UCRCorpusDataset(
    DATA_PATH, per_dataset_cap=2000,
    transform=ContrastiveTransformations(ts_aug, n_views=2),
)
print(f"\nContrastive training pool: {len(train_corpus)} series (labels discarded)")

train_loader = data.DataLoader(
    train_corpus, batch_size=256, shuffle=True, drop_last=True, num_workers=0, pin_memory=True
)

## Model: patch-Transformer encoder + projection head

Following SimCLR, the model is a base encoder $f(\cdot)$ -- our shared patch Transformer -- and
a small projection head $g(\cdot)$, a 2-layer MLP with ReLU. The contrastive loss is applied to
$g(f(x))$; after pretraining $g$ is **discarded** and the encoder's `[CLS]`-based global feature
is the representation we probe.

Discarding the head is not a detail. The projection head absorbs the augmentation-specific
information the loss needs but the downstream task does not, which is why the representation
*before* the head consistently probes better than the one after it.

In [ ]:
class SimCLRTSModel(nn.Module):
    def __init__(self, encoder, embed_dim=128, hidden_dim=256, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.projection_head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

    def forward(self, x):
        h = self.encoder.forward_features(x, pool="cls")  # (B, embed_dim)
        z = self.projection_head(h)                        # (B, proj_dim)
        return z

## The InfoNCE loss

Given a batch of `2N` embeddings (`N` series x 2 views, concatenated), the implementation
below:

1. computes pairwise **cosine similarity** between all `2N` embeddings;
2. masks out each embedding's similarity to **itself**;
3. builds a mask locating each embedding's **positive** (the other view of the same series);
4. scales by temperature `tau` and computes the InfoNCE negative log-likelihood.

<div class="alert alert-info">

**Why the positive mask works:** with `n_views=2` and the two views of series `k` at
positions `k` and `k + N` in the concatenated batch, the positive of position `i` sits at
`(i + N) mod 2N` -- exactly what `torch.eye(...).roll(shifts=N, dims=0)` produces. This is
the *same* cell as the remote sensing notebook 1.

</div>

In [ ]:
def info_nce_loss(feats, temperature=0.1):
    '''
    feats: (2N, proj_dim) embeddings, views of the same series at offset N apart.
    Returns: (scalar loss, cos_sim matrix, pos_mask) for logging.
    '''
    # TODO: info_nce
    raise NotImplementedError("Fill in info_nce -- see the notebook markdown above.")
    return loss, cos_sim, pos_mask


def info_nce_metrics(cos_sim, pos_mask):
    '''Ranking diagnostics: top-1 / top-5 accuracy of the positive among all negatives.'''
    comb_sim = torch.cat(
        [cos_sim[pos_mask][:, None], cos_sim.masked_fill(pos_mask, -9e15)], dim=-1
    )
    sim_argsort = comb_sim.argsort(dim=-1, descending=True).argmin(dim=-1)
    acc_top1 = (sim_argsort == 0).float().mean().item()
    acc_top5 = (sim_argsort < 5).float().mean().item()
    return acc_top1, acc_top5

## Live training (dynamics only)

<div class="alert alert-info">

**What this cell shows vs. what we evaluate later.** Full contrastive pretraining needs many
passes over the corpus. Here we run a **short** live loop so you can *see the InfoNCE loss fall
and top-1/top-5 accuracy rise*. The checkpoint loaded afterward was pretrained **offline for
15,000 steps** on the same corpus
(`src/time_series/pretraining/train_contrastive_ts.py`).

Watch the top-1 accuracy: picking the right positive out of 511 candidates by chance is
$1/511 \approx 0.2\%$, so anything meaningfully above that means the encoder is already
learning something augmentation-invariant.

</div>

In [ ]:
model = SimCLRTSModel(build_ts_encoder()).to(device)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

NUM_STEPS = 300
TEMPERATURE = 0.1

loss_history, acc1_history = [], []
data_iter = iter(train_loader)
model.train()

for step in tqdm(range(NUM_STEPS)):
    try:
        views = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        views = next(data_iter)

    series = torch.cat(views, dim=0).to(device)  # (2N, 128, 1)
    feats = model(series)
    loss, cos_sim, pos_mask = info_nce_loss(feats, temperature=TEMPERATURE)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    acc1, _ = info_nce_metrics(cos_sim.detach(), pos_mask)
    loss_history.append(loss.item())
    acc1_history.append(acc1)

print(f"InfoNCE loss {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")
print(f"top-1 accuracy {acc1_history[0]:.3f} -> {acc1_history[-1]:.3f} "
      f"(chance = {1/(2*256-1):.4f})")

plot_curve(loss_history, xlabel="Step", ylabel="InfoNCE loss",
           title="Live contrastive training (dynamics only)")
plot_curve(acc1_history, xlabel="Step", ylabel="Top-1 accuracy (positive vs. negatives)",
           title="Positive-pair ranking accuracy")

## Loading the pretrained encoder

We load the checkpoint pretrained offline on the pooled UCR corpus. If it is not hosted yet, we
fall back to the briefly-trained model above so the rest of the notebook still runs.

In [ ]:
pretrained_model = SimCLRTSModel(build_ts_encoder()).to(device)
found = try_load_checkpoint(pretrained_model.encoder, CHECKPOINT_PATH,
                            "contrastive_ts_encoder.pt", device)
if not found:
    print("Falling back to the briefly-trained model from the live-training cell above.")
    pretrained_model = model

pretrained_model.eval()
encoder = pretrained_model.encoder
for p in encoder.parameters():
    p.requires_grad = False
print("Encoder frozen for evaluation.")

## Does it classify? The frozen linear probe

This is the moment the tutorial has been building toward, and it is *identical* to the remote sensing part's
evaluation: freeze the encoder, extract one feature vector per series, fit a logistic
regression, report test accuracy.

The encoder has **never seen a SwedishLeaf label**. It was trained only to tell augmented views
of the same series apart from other series, on a corpus in which SwedishLeaf is about 1% of the
data.

In [ ]:
train_ds = UCRDataset(DATA_PATH, TARGET_DATASET, "train")
test_ds  = UCRDataset(DATA_PATH, TARGET_DATASET, "test")
class_names = ucr_class_names(DATA_PATH, TARGET_DATASET)

tr_f, tr_y = extract_features(encoder, train_ds, device, pool="cls")
te_f, te_y = extract_features(encoder, test_ds,  device, pool="cls")

contrastive_acc = linear_probe_accuracy(tr_f, tr_y, te_f, te_y)
print(f"Contrastive frozen linear probe on {TARGET_DATASET}: {contrastive_acc:.4f}")

# The random-init floor from Notebook 0, recomputed here so the comparison is self-contained.
seed_everything(123)
rand_enc = build_ts_encoder().to(device).eval()
for p in rand_enc.parameters():
    p.requires_grad = False
r_tr, r_try = extract_features(rand_enc, train_ds, device, pool="cls")
r_te, r_tey = extract_features(rand_enc, test_ds,  device, pool="cls")
random_acc = linear_probe_accuracy(r_tr, r_try, r_te, r_tey)
print(f"Random-init floor:                              {random_acc:.4f}")
print(f"\nContrastive pretraining gain: {contrastive_acc - random_acc:+.4f}")

## Visualizing the learned embedding space

A linear probe gives one number; a t-SNE plot shows *why* that number is what it is. We reduce
the frozen 128-dim `[CLS]` embeddings of the held-out test series to 2D and color each point by
its true class.

Remember: the encoder never saw these labels. Any class structure visible here was discovered
purely from the instance-discrimination objective.

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, init="pca", perplexity=30)
embedding_2d = tsne.fit_transform(te_f)

plot_embedding_scatter(embedding_2d, te_y, class_names=class_names,
                       title=f"t-SNE of contrastive embeddings -- {TARGET_DATASET} test series")

**Reading the scatter plot.** If contrastive pretraining learned something
class-relevant, series should organize by leaf species even though species was never a training
signal. Look for compact same-color groups and for which classes bleed into each other -- those
confusions are exactly the ones the linear probe pays for.

The augmentations declared jitter, mild scaling, level shifts and short time-masks to be
**nuisance**; whatever structure survives that is the shape information the encoder chose to
keep.

<div class="alert alert-info">

**Aside -- beyond generic augmentations.** TS2Vec and related methods add time-series-specific
positives: *neighboring* sub-windows of the same series (temporal contrast), or the same
series seen in the time domain vs. the frequency domain (TF-C). These are stronger,
free-standing invariance signals -- the analogue of SeCo's seasonal positives for imagery.

</div>

### References

- Yue, Z. et al. (2022). *TS2Vec: Towards Universal Representation of Time Series.* AAAI.
- Zhang, X. et al. (2022). *Self-Supervised Contrastive Pre-Training for Time Series via
  Time-Frequency Consistency (TF-C).* NeurIPS.
- Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). *A Simple Framework for
  Contrastive Learning of Visual Representations (SimCLR).* ICML.
- Nie, Y. et al. (2023). *A Time Series is Worth 64 Words: Long-term Forecasting with
  Transformers (PatchTST).* ICLR.
- Dau, H. A. et al. (2019). *The UCR Time Series Archive.* IEEE/CAA J. Autom. Sinica.
- Stival, L. et al. (2025/2026). *[Tutorial paper -- full citation TBD].*